## Post Process PLY

In [1]:
from plyfile import PlyData, PlyElement
import numpy as np

ply_file = "/home/yeliu/Development/ADATA/pcl/nanchansi_day_v2.ply"
output_ply_file = "/home/yeliu/Development/ADATA/pcl/nanchansi_day_v2_proc.ply"

In [4]:
def filter_ply_by_scale(ply_file, output_ply_file):
    scale_max = 1.0
    opacity_min = -2.0
    plydata = PlyData.read(ply_file)
    
    vertices = plydata['vertex']
    origin_size = len(vertices)
    
    # filter by scale
    scale_xyz = np.stack((np.asarray(vertices["scale_0"]),
                          np.asarray(vertices["scale_1"]),
                          np.asarray(vertices["scale_2"])),  axis=1)    
    mask_scale = scale_xyz.max(axis=1) <= scale_max
    final_mask = mask_scale
    
    # filter by opacity
    value_opacities = np.asarray(vertices["opacity"])
    mask_opacities = value_opacities > opacity_min
    
    final_mask = final_mask & mask_opacities

    # filter by color
#     SH_C0 = 0.28209479177387814;
#     color_r = 0.5 + SH_C0 * np.asarray(vertices["f_dc_0"])
#     color_g = 0.5 + SH_C0 * np.asarray(vertices["f_dc_1"])
#     color_b = 0.5 + SH_C0 * np.asarray(vertices["f_dc_2"])
#     mask_r = (color_r > 0.0) & (color_r < 1.0)
#     mask_g = (color_g > 0.0) & (color_g < 1.0)
#     mask_b = (color_b > 0.0) & (color_b < 1.0)
#     final_mask = final_mask & mask_r & mask_g & mask_b
    
    filtered_vertices = vertices[final_mask]
    final_size = len(filtered_vertices)
    
    print(f"save {origin_size} -> {final_size}")
    new_plydata = PlyData([PlyElement.describe(filtered_vertices, 'vertex')])
    new_plydata.write(output_ply_file)    

In [5]:
filter_ply_by_scale(ply_file, output_ply_file)

save 9547806 -> 5913739


## Post Process Image before SSIM

In [18]:
from PIL import Image
import torch
import numpy as np
from torchmetrics.image.ssim import StructuralSimilarityIndexMeasure

def read_test_image(image_path, ratio=0.25):
    image = Image.open(image_path)
    if ratio != 1.0:
        image = image.resize((int(image.size[0] * ratio), int(image.size[1] * ratio)), Image.Resampling.LANCZOS)

    image_torch = torch.from_numpy(np.array(image)) / 255.0
    image_torch = image_torch.permute(2, 0, 1).unsqueeze(dim=0)
    return image_torch

def torch_to_pil(tensor):
    # Ensure the tensor is on CPU and has the correct shape
    tensor = tensor.squeeze(0)  # Remove the batch dimension
    tensor = tensor.permute(1, 2, 0)  # Change shape from [C, H, W] to [H, W, C]
    tensor = tensor.numpy()  # Convert to NumPy array

    # Convert to uint8 and scale to [0, 255]
    tensor = (tensor * 255).astype(np.uint8)

    # Create a PIL image
    pil_image = Image.fromarray(tensor)
    return pil_image

* Test with night image : `SSIM Score: 0.6718. L1 Score: 0.0589`
* Test with day image : `SSIM Score: 0.5820. L1 Score: 0.0732`

In [20]:
image_torch_test = read_test_image("../../data/night_ssim_test.jpg", 1.0)
image_torch_gt = read_test_image("../../data/night_ssim_gt.jpg", 1.0)

# Initialize SSIM metric
ssim_metric = StructuralSimilarityIndexMeasure(data_range=1.0)
ssim_score = ssim_metric(image_torch_test, image_torch_gt)

l1_loss = torch.abs((image_torch_test - image_torch_gt)).mean()
print(f"SSIM Score: {ssim_score.item():.4f}. L1 Score: {l1_loss.item():.4f}")


# process and show image


image_pil = torch_to_pil(image_torch_test)
# display(image_pil)

SSIM Score: 0.6718. L1 Score: 0.0589
